In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"font.size": 12})
%matplotlib widget
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import c, eV, m_e, micro, nano

sigmaz = 300 * micro
sigmax = 500 * nano
sigmay = 10 * nano
binsz = np.linspace(-8 * sigmaz, 8 * sigmaz, 129)

GeV = 1e9 * eV
# gp
Nz = 16
N_steps = 2 * Nz - 1
cut_z = 4 * sigmaz
T = 2 * cut_z / c
dt = T / N_steps
print(N_steps, dt)

In [ ]:
fig, ax = plt.subplots(ncols=7, nrows=2, figsize=(20, 8), dpi=100)

it = N_steps - 1

x_list = []
y_list = []
z_list = []
w_list = []
ux_list = []
uy_list = []
uz_list = []

# Loop through the files that contain particles collected at the edges and in the box
for species in ("ele", "pos"):
    for folder_name in [
        "./diags/bound/particles_at_xlo",
        "./diags/bound/particles_at_xhi",
        "./diags/bound/particles_at_ylo",
        "./diags/bound/particles_at_yhi",
        "./diags/diag1",
    ]:
        series = OpenPMDTimeSeries(folder_name)
        x, y, z, w, ux, uy, uz = series.get_particle(
            ["x", "y", "z", "w", "ux", "uy", "uz"], iteration=it, species=species
        )
        x_list.append(x)
        y_list.append(y)
        z_list.append(z)
        w_list.append(w)
        ux_list.append(ux)
        uy_list.append(uy)
        uz_list.append(uz)

# Concatenate list of particles from all files
x_all = np.concatenate(x_list)
y_all = np.concatenate(y_list)
z_all = np.concatenate(z_list)
ux_all = np.concatenate(ux_list)
uy_all = np.concatenate(uy_list)
uz_all = np.concatenate(uz_list)
w_all = np.concatenate(w_list)
gamma = np.sqrt(1 + ux_all**2 + uy_all**2 + uz_all**2)
energy = m_e * c**2 * gamma / GeV

print(f"number of macropairs in warpx {len(x_all)}")
# ax[0][0].scatter(z_all,x_all)
# ax[1][0].scatter(z_all,y_all)

print("x", x_all.min(), x_all.max())
print("y", y_all.min(), y_all.max())
print("z", z_all.min(), z_all.max())
print("ux", (ux_all / gamma).min(), (ux_all / gamma).max())
print("ux", (ux_all / gamma).min(), (ux_all / gamma).max())
print("uz", (uz_all / gamma).min(), (uz_all / gamma).max())
print("e", energy.min(), energy.max(), np.average(energy))

H, b = np.histogram(energy, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][0].plot(b, H)
ax[1][0].set(title="wx energy", yscale="log")


H, b = np.histogram(ux_all / gamma, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][1].plot(b, H)
ax[1][1].set(title="wx ux")

H, b = np.histogram(uy_all / gamma, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][2].plot(b, H)
ax[1][2].set(title="wx uy")


H, b = np.histogram(uz_all / gamma, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][3].plot(b, H)
ax[1][3].set(title="wx uz")


H, b = np.histogram(x_all, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][4].plot(b, H)
ax[1][4].set(title="wx x")

H, b = np.histogram(y_all, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][5].plot(b, H)
ax[1][5].set(title="wx y")

H, b = np.histogram(z_all, bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[1][6].plot(b, H)
ax[1][6].set(title="wx z")


#    energy of the particle (GeV), E1. Positive for electron, negative for positron.
#    normalized velocity in x direction (vx/c = px/E)
#    normalized velocity in y direction (vy/c = py/E)
#    normalized velocity in z direction (vz/c = pz/E)
#    process label : 0 = Breit-Wheeler ; 1 = Bethe-Heitler ; 2 = Landau-Lifschitz
#    a label, individuating each pair

pairs = np.genfromtxt("gp/pairs.dat")
x = pairs[:, 4] * 1e-6  # Convert to m
y = pairs[:, 5] * 1e-6  # Convert to m
z = pairs[:, 6] * 1e-9  # Convert to m
energy = pairs[:, 0]
print("average energy guinea ", np.average(np.abs(energy)))
print(f"number of macropairs in guinea {len(energy)}")

H, b = np.histogram(np.abs(pairs[:, 0]), bins=101, density=True)
b = 0.5 * (b[1:] + b[:-1])
ax[0][0].plot(b, H)
ax[0][0].set(title="gp energy", yscale="log")

for i in range(1, 7):
    print(i, pairs[:, i].min(), pairs[:, i].max())

    H, b = np.histogram(pairs[:, i], bins=101, density=True)
    b = 0.5 * (b[1:] + b[:-1])
    ax[0][i].plot(b, H)
    ax[0][i].set(title=f"gp {i}")


fig.tight_layout()
fig.savefig("pairs.png", dpi=300)
plt.show()
plt.close()

In [ ]:
for i in range(30):
    fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(10, 10), dpi=100, sharey="row")

    series = OpenPMDTimeSeries("./diags/diag1/")

    it = series.iterations[i]
    x2, y2, z2 = series.get_particle(["x", "y", "z"], species="beam1", iteration=it)

    x1, y1, z1 = series.get_particle(["x", "y", "z"], species="beam2", iteration=it)

    print(f"number of macroparticles in warpx {len(x1)}, {len(x2)}")
    ax[0][0].scatter(z1, x1)
    ax[0][0].scatter(z2, x2)

    ax[1][0].scatter(z1, y1)
    ax[1][0].scatter(z2, y2)

    # Particle Energy [GeV] | x [um] | y [um] | z [um] | x' [urad] | y' [urad]
    beam1 = np.loadtxt(f"gp/b1.{i:g}")
    beam2 = np.loadtxt(f"gp/b2.{i:g}")
    x1 = beam1[:, 1] * 1e-6  # Convert to m
    y1 = beam1[:, 2] * 1e-6  # Convert to m
    z1 = beam1[:, 3] * 1e-6  # Convert to m
    x2 = beam2[:, 1] * 1e-6  # Convert to m
    y2 = beam2[:, 2] * 1e-6  # Convert to m
    z2 = beam2[:, 3] * 1e-6  # Convert to m

    ax[0][1].scatter(z1, x1)
    ax[0][1].scatter(z2, x2)

    ax[1][1].scatter(z1, y1)
    ax[1][1].scatter(z2, y2)

    print(f"number of macroparticles in guinea {len(x1)}, {len(x2)}")

    for a in ax.reshape(-1):
        a.axvline(8 * sigmaz, color="gray")
        a.axvline(-8 * sigmaz, color="gray")

    for a in ax[0, :].reshape(-1):
        a.axhline(8 * sigmax, color="gray")
        a.axhline(-8 * sigmax, color="gray")

    for a in ax[1, :].reshape(-1):
        a.axhline(8 * sigmay, color="gray")
        a.axhline(-8 * sigmay, color="gray")

    fig.tight_layout()
    fig.savefig(f"beams_{i:g}.png", dpi=300)
    # plt.show()
    plt.close()